# Stage 2: Algerian Darija <-> English translation fine-tuning
Continues fine-tuning the Stage 1 LoRA adapter on bidirectional translation, using explicit translation instructions in every example so the model gets used to acting as a translation assistant.

In [1]:
!pip install -q \
    "transformers==5.16.1" \
    "datasets" \
    "peft" \
    "trl" \
    "accelerate" \
    "bitsandbytes" \
    "huggingface_hub" \
    "scikit-learn"
print("Packages installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 87.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 83.0 MB/s eta 0:00:00:00:01
Packages installed.


In [2]:
import transformers

print(transformers.__version__)

5.16.1


In [3]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoConfig, BitsAndBytesConfig, AutoModelForCausalLM, TextStreamer, TrainingArguments, Trainer
from huggingface_hub import login
from sklearn.model_selection import train_test_split
from datasets import Dataset
from kaggle_secrets import UserSecretsClient
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,
    PeftModelForCausalLM
)
from trl import SFTTrainer

In [ ]:
# login to Hugging Face using the token from the .env file
# from dotenv import load_dotenv
# load_dotenv()
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
if hf_token:
    login(token=hf_token)
else:
    print("Warning: HF_TOKEN not found in environment")
model_id = "google/gemma-4-E2B"
stage1_adapter_dir = "/kaggle/input/datasets/khalilghanem/gemma4-darija-qlora-stage1"  # path to your Stage 1 adapter dataset
device_map = {"": 0}
device = "cuda" if torch.cuda.is_available() else "cpu"

# T4 (Turing) has NO bf16 tensor cores -> bf16 runs as slow emulated math (~50-100x slower).
# Force fp16 everywhere: it uses the T4's fp16 tensor cores and is the correct fast path here.
compute_dtype = torch.float16
print(f"Base model: {model_id}")
print(f"Stage 1 adapter: {stage1_adapter_dir}")
print(f"Compute dtype: {compute_dtype}")

In [5]:
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")

GPU: Tesla T4
VRAM: 14.56219482421875 GB


In [6]:
# Quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=compute_dtype,  # fp16 - see note above on T4/bf16
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["vision_tower", "audio_tower"],
)
print("Quantization config ready.")

Quantization config ready.


In [7]:
print(torch.cuda.memory_allocated() / 1024**3, "GB allocated")
print(torch.cuda.memory_reserved() / 1024**3, "GB reserved")

0.0 GB allocated
0.0 GB reserved


In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_id, extra_special_tokens={})

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

streamer = TextStreamer(tokenizer, skip_prompt=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map={'': 0},
    dtype=compute_dtype,
    attn_implementation="sdpa",  # Accelerated PyTorch Scaled Dot-Product Attention
)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/906 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [9]:
print("Model loaded successfully. Skipping full model print to save memory.")

Model loaded successfully. Skipping full model print to save memory.


In [ ]:
model.config.use_cache = False

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

# Selectively upcast small non-4bit params to float32
# (skip large embeddings that would exceed 8GB VRAM)
SKIP_UPCAST = {"embed_tokens_per_layer", "embed_tokens", "lm_head"}
for name, param in model.named_parameters():
    if param.__class__.__name__ == "Params4bit":
        continue  # quantized weights - leave as-is
    if param.dtype not in (torch.float16, torch.bfloat16):
        continue  # already float32 or other dtype
    if any(skip in name for skip in SKIP_UPCAST):
        continue  # too large to safely upcast on 8GB VRAM
    param.data = param.data.to(torch.float32)

# Step 4: Freeze all base-model parameters
for param in model.parameters():
    param.requires_grad_(False)

# Crucial for gradient checkpointing when prepare_model_for_kbit_training is skipped:
# Ensures input activations track gradients into the adapter layers
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
else:
    def make_inputs_require_grad(module, input, output):
        output.requires_grad_(True)
    model.get_input_embeddings().register_forward_hook(make_inputs_require_grad)

import gc
gc.collect()
torch.cuda.empty_cache()

print(f"VRAM after prep: {torch.cuda.memory_allocated() / 1024**3:.2f} GB allocated")

# Load the Stage 1 adapter and keep training it (instead of creating a fresh LoRA)
model = PeftModelForCausalLM.from_pretrained(model, stage1_adapter_dir, is_trainable=True)

# Force all trainable (LoRA) params to fp32 master weights.
# The Stage-1 adapter checkpoint was saved in bf16 (confirmed). If any trainable
# param stays bf16 while fp16=True (GradScaler) is used, training crashes with:
# "_amp_foreach_non_finite_check_and_unscale_cuda not implemented for 'BFloat16'"
# Casting adapter weights to fp32 here (they are tiny, so memory cost is negligible)
# makes them compatible with the fp16 GradScaler and removes the crash for good.
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

# Guard: EVERY trainable param must be fp32, otherwise the fp16 GradScaler crashes with
# "_amp_foreach_non_finite_check_and_unscale_cuda not implemented for 'BFloat16'".
# Fail loudly here (before training) instead of mid-run.
_bad = [(n, str(p.dtype)) for n, p in model.named_parameters()
        if p.requires_grad and p.dtype != torch.float32]
assert not _bad, f"Trainable params not fp32 (fp16 training would crash): {_bad[:5]}"

model.print_trainable_parameters()

In [ ]:
# --- Sanity diagnostic: confirm dtypes + device are clean BEFORE training ---
# (The whole slow/crash saga came from dtype confusion; this makes the state explicit.)
import collections

trainable = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
frozen = [(n, p) for n, p in model.named_parameters() if not p.requires_grad]

def _dtype_hist(params):
    h = collections.Counter()
    for _, p in params:
        h[str(p.dtype)] += p.numel()
    return {k: f"{v/1e6:.1f}M" for k, v in h.items()}

print("Trainable param dtypes:", _dtype_hist(trainable))
print("Frozen param dtypes:   ", _dtype_hist(frozen))
print("Param devices:         ", {str(p.device) for _, p in model.named_parameters()})

# Hard guarantees for the fast fp16 path on T4:
assert all(p.dtype == torch.float32 for _, p in trainable), "trainable params must be fp32"
assert all(str(p.device).startswith("cuda") for _, p in model.named_parameters()), "everything must be on GPU"
print("\nOK -> trainable=fp32, frozen 4-bit base computes in fp16, all on GPU. Ready to train.")


## Build the bidirectional translation dataset
Each CSV row (`id, Arabic, English`) produces **two** training examples: Darija -> English and English -> Darija. The instruction is always written in the same language as the input, and it stays identical to what we'll use at inference time.

In [ ]:
EN_TO_AR_TEMPLATE = """ترجم الجملة التالية من الإنجليزية إلى الدارجة الجزائرية:

{sentence}

الترجمة: """

AR_TO_EN_TEMPLATE = """Translate the following Algerian Darija sentence to English:

{sentence}

Translation: """

# Max sequence length (tokens). Examples longer than this are DROPPED, not truncated,
# so we never teach the model to emit half-finished translations. ~95% of pairs fit in 512.
MAX_SEQ_LEN = 512

print("Loading translation CSV...")
df = pd.read_csv("/kaggle/input/datasets/khalilghanem/translation-data/algerian_translation_50k_cleaned.csv")  # id, Arabic, English
df = df.dropna(subset=["Arabic", "English"])
print(f"Loaded {len(df)} rows.")

print("Building bidirectional examples...")
darija_list = df["Arabic"].astype(str).str.strip().tolist()
english_list = df["English"].astype(str).str.strip().tolist()
eos = tokenizer.eos_token
texts = []
for darija, english in zip(darija_list, english_list):
    texts.append(EN_TO_AR_TEMPLATE.format(sentence=english) + darija + eos)  # English -> Darija
    texts.append(AR_TO_EN_TEMPLATE.format(sentence=darija) + english + eos)  # Darija -> English

dataset = Dataset.from_pandas(pd.DataFrame({"text": texts}))
print(f"Built {len(dataset)} raw examples.")

# Drop examples that exceed MAX_SEQ_LEN tokens (avoid truncating translation targets).
def _fits(batch):
    enc = tokenizer(batch["text"], add_special_tokens=True)
    return [len(ids) <= MAX_SEQ_LEN for ids in enc["input_ids"]]

before = len(dataset)
dataset = dataset.filter(_fits, batched=True, num_proc=4)
print(f"Kept {len(dataset)}/{before} examples after {MAX_SEQ_LEN}-token length filter "
      f"({100 * len(dataset) / before:.1f}%).")

splits = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = splits["train"]
# Cap eval set so periodic evaluation stays cheap (the full 5% would make each eval
# nearly as expensive as ~100 training steps).
eval_dataset = splits["test"].select(range(min(1000, len(splits["test"]))))
print(f"{len(train_dataset)} train / {len(eval_dataset)} eval examples.")


In [12]:
print(train_dataset[0]["text"])

Translate the following Algerian Darija sentence to English:

في مقدمة القارب، حط زوج فيلي تاع الحوت فوق الخشب ومعاهم الحوت الطائر.

Translation: Back in the bow, he laid the two fillets of fish out on the wood with the flying fish beside them.<eos>


In [ ]:
import dataclasses
import inspect
from trl import SFTConfig

output_dir = "/kaggle/working/gemma4-darija-en-translation-qlora"

# Precision note: T4 (Turing) has NO bf16 tensor cores, so bf16 falls back to slow
# emulated math (~50-100x slower). fp16 uses the T4's fp16 tensor cores and is the
# fast path. LoRA params are fp32, so fp16's GradScaler runs without the bf16 crash.
#
# packing=False: the T4 has no FlashAttention, and packing + sdpa cross-contaminates
# samples (attention bleeds across unrelated translation pairs). Padding waste is
# instead handled by the length-grouped sampler in the next cell.
cfg_kwargs = dict(
    output_dir=output_dir,
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,         # long examples were dropped, not truncated, above
    packing=False,
    dataset_num_proc=4,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,  # effective batch size = 16
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    num_train_epochs=1,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    fp16=True,
    bf16=False,
    fp16_full_eval=False,
    optim="adamw_8bit",             # non-paged: paged optims can stall on CPU paging
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
)

# transformers 5.x / TRL 1.x dropped some TrainingArguments fields (group_by_length is
# gone). Filter to what this installed version accepts instead of crashing on it.
_valid = ({f.name for f in dataclasses.fields(SFTConfig)} if dataclasses.is_dataclass(SFTConfig)
            else set(inspect.signature(SFTConfig.__init__).parameters))
_dropped = sorted(k for k in cfg_kwargs if k not in _valid)
if _dropped:
    print(f"NOTE: dropped unsupported SFTConfig args for this version: {_dropped}")
cfg_kwargs = {k: v for k, v in cfg_kwargs.items() if k in _valid}

# Keep warmup even if warmup_ratio was renamed to warmup_steps.
if "warmup_ratio" not in cfg_kwargs and "warmup_steps" in _valid:
    cfg_kwargs["warmup_steps"] = 100

training_args = SFTConfig(**cfg_kwargs)
print("Training config ready. Output dir:", output_dir)
print(f"fp16={training_args.fp16}  bf16={training_args.bf16}  optim={training_args.optim}")

In [ ]:
# --- Precision hygiene: force accelerate onto the fp16 path ---
#
# Confirmed empirically: the gradients arrive as bf16 (a grad hook returning fp32
# raised "hook has changed the type of value (was CUDABFloat16Type)"). bf16 grads +
# an fp16 GradScaler is exactly what makes unscale_() raise
# "_amp_foreach_non_finite_check_and_unscale_cuda not implemented for 'BFloat16'".
#
# So the weights were never the problem - the autocast dtype is. Two things pin it:
#   1. ACCELERATE_MIXED_PRECISION in the environment overrides the TrainingArguments.
#   2. AcceleratorState is a process-global singleton: a Trainer built earlier in
#      this kernel with bf16=True leaves mixed_precision pinned to "bf16", and a new
#      Trainer with fp16=True does NOT reset it.
# Force both to fp16 before the Trainer (and its Accelerator) is constructed.
import os

from accelerate.state import AcceleratorState

for _var in ("ACCELERATE_MIXED_PRECISION", "ACCELERATE_DOWNCAST_BF16"):
    if os.environ.get(_var):
        print(f"overriding {_var}={os.environ[_var]!r}")
os.environ["ACCELERATE_MIXED_PRECISION"] = "fp16"

try:
    _before = AcceleratorState().mixed_precision
except Exception:
    _before = "(uninitialised)"
try:
    AcceleratorState._reset_state(reset_partial_state=True)
    print(f"Accelerator state reset (was mixed_precision={_before!r}).")
except Exception as _e:
    print(f"Accelerator reset skipped ({_e}); restart the kernel if this persists.")

# --- Trainer + the fp32 adapter fix ---
#
# ROOT CAUSE (measured, not guessed): the trainable LoRA params are bfloat16 at
# training time - 24.2M of them, matching the 410 bf16 tensors in the Stage-1
# checkpoint. Autocast is correctly fp16 and the frozen base is fp16, but under
# fp16 autocast a bf16 WEIGHT yields a bf16 GRADIENT, and the fp16 GradScaler then
# raises "_amp_foreach_non_finite_check_and_unscale_cuda not implemented for
# 'BFloat16'" inside unscale_().
#
# The cast in the model-prep cell does happen, but something in the PEFT/TRL init
# path restores the adapters to the checkpoint dtype afterwards. So cast twice:
# once right after the Trainer is built, and once more from on_train_begin, which
# runs AFTER the optimizer is created and is the last point a cast can take effect.
# .data is mutated in place, so the optimizer keeps referencing the same Parameter
# objects either way.
import random
import time
import numpy as np
from torch.utils.data import Sampler
from transformers import TrainerCallback


def force_fp32_trainable(module, tag=""):
    """Cast every trainable param to fp32 in place. Returns how many were changed."""
    names = [n for n, p in module.named_parameters() if p.requires_grad and p.dtype != torch.float32]
    for _, p in module.named_parameters():
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.float()
    if names:
        print(f"[fp32{tag}] re-cast {len(names)} trainable tensors (e.g. {names[0]})")
    return len(names)


def assert_trainable_fp32(module):
    bad = [(n, str(p.dtype)) for n, p in module.named_parameters() if p.requires_grad and p.dtype != torch.float32]
    assert not bad, f"trainable params still not fp32: {bad[:3]}"


class ForceFp32Adapters(TrainerCallback):
    """Last line of defence: re-cast adapters to fp32 once training actually starts."""

    def on_train_begin(self, args, state, control, model=None, **kwargs):
        force_fp32_trainable(model, tag=" on_train_begin")
        assert_trainable_fp32(model)


class LengthGroupedSampler(Sampler):
    """Batches similar-length examples together.

    `group_by_length` does not exist in this transformers/TRL version, but this
    dataset has a heavy length tail (median ~73 tokens, p95 ~300), so random
    batching pads nearly every sequence up to the longest member of its batch -
    roughly 3x wasted compute. Strategy: shuffle, cut into megabatches, sort each
    megabatch by length, then shuffle batch order. Near-random sampling, minimal
    padding, and no packing-style cross-contamination.
    """

    def __init__(self, lengths, batch_size, seed=42, megabatch_mult=50):
        self.lengths = np.asarray(lengths)
        self.batch_size = batch_size
        self.seed = seed
        self.mega = batch_size * megabatch_mult

    def __len__(self):
        return len(self.lengths)

    def __iter__(self):
        rng = np.random.default_rng(self.seed)
        order = rng.permutation(len(self.lengths))
        batches = []
        for start in range(0, len(order), self.mega):
            chunk = order[start:start + self.mega]
            chunk = chunk[np.argsort(self.lengths[chunk], kind="stable")]
            batches += [chunk[i:i + self.batch_size] for i in range(0, len(chunk), self.batch_size)]
        random.Random(self.seed).shuffle(batches)
        return iter(int(i) for b in batches for i in b)


class LengthGroupedSFTTrainer(SFTTrainer):
    """SFTTrainer that batches by length (replaces the removed group_by_length arg)."""

    _cached_lengths = None

    def _get_train_sampler(self, *args, **kwargs):
        ds = self.train_dataset
        if ds is None or "input_ids" not in getattr(ds, "column_names", []):
            return super()._get_train_sampler(*args, **kwargs)
        if self._cached_lengths is None:
            self._cached_lengths = [len(ids) for ids in ds["input_ids"]]
            mean_len = np.mean(self._cached_lengths)
            p95_len = np.percentile(self._cached_lengths, 95)
            print(f"[sampler] length-grouped over {len(self._cached_lengths)} examples "
                            f"(mean {mean_len:.0f}, p95 {p95_len:.0f} tokens)")
        return LengthGroupedSampler(
            self._cached_lengths,
            batch_size=self.args.per_device_train_batch_size,
            seed=self.args.seed,
        )


class SpeedCallback(TrainerCallback):
    """Prints wall-clock seconds per optimizer step for the first few steps, so the
    fast fp16 path is confirmed immediately: expect a few s/step, NOT the ~50s/step
    the bf16 and no-autocast runs produced."""

    def __init__(self, report_first=8):
        self.report_first = report_first
        self._t = None

    def on_step_end(self, args, state, control, **kwargs):
        now = time.time()
        if self._t is not None and state.global_step <= self.report_first:
            dt = now - self._t
            print(f"[speed] step {state.global_step}: {dt:.2f}s/step ({1.0 / dt:.2f} it/s)")
        self._t = now


# model already carries the Stage 1 LoRA adapter (is_trainable=True above).
trainer = LengthGroupedSFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    args=training_args,
    callbacks=[ForceFp32Adapters(), SpeedCallback()],
)

# Cast #1: immediately after Trainer init (this is where the revert to bf16 happens).
force_fp32_trainable(trainer.model, tag=" post-init")
assert_trainable_fp32(trainer.model)
print("Trainer ready, adapters are fp32. Run the smoke test next.")

In [ ]:
# --- Precision smoke test: settle the bf16 question in ~20s, before a multi-hour run ---
# Prints every input that decides the autocast dtype, then runs one real
# forward/backward and reports the actual gradient dtypes.
import collections
import os

print("--- what the config says ---")
print("training_args.fp16          :", training_args.fp16)
print("training_args.bf16          :", training_args.bf16)
print("half_precision_backend      :", getattr(training_args, "half_precision_backend", "n/a"))
print("ACCELERATE_MIXED_PRECISION  :", os.environ.get("ACCELERATE_MIXED_PRECISION"))
print("accelerator.mixed_precision :", trainer.accelerator.mixed_precision)
print("scaler                      :", type(getattr(trainer.accelerator, "scaler", None)).__name__)

print("\n--- what the model actually holds ---")
_h = collections.Counter()
for _n, _p in model.named_parameters():
    _h[("trainable" if _p.requires_grad else "frozen", str(_p.dtype))] += _p.numel()
for (_kind, _dt), _cnt in sorted(_h.items()):
    print(f"  {_kind:<9} {_dt:<16} {_cnt/1e6:8.1f}M")

print("\n--- one real forward/backward ---")
batch = next(iter(trainer.get_train_dataloader()))
batch = {k: (v.to(model.device) if hasattr(v, "to") else v) for k, v in batch.items()}
model.train()
with trainer.accelerator.autocast():
    out = model(**batch)
out.loss.backward()

grad_dtypes = collections.defaultdict(list)
for _n, _p in model.named_parameters():
    if _p.grad is not None:
        grad_dtypes[str(_p.grad.dtype)].append(_n)

print("loss:", out.loss.detach().item())
for _dt, _names in grad_dtypes.items():
    print(f"  grads {_dt}: {len(_names)} tensors  e.g. {_names[0]}")

model.zero_grad(set_to_none=True)
del out, batch
gc.collect()
torch.cuda.empty_cache()

_nonfp32 = sorted(d for d in grad_dtypes if d != "torch.float32")
if _nonfp32:
    print(f"\nFAIL: grads are {_nonfp32} but the fp16 GradScaler requires fp32.")
    print("      trainer.train() would crash in unscale_(). Do not start the long run.")
elif trainer.accelerator.mixed_precision != "fp16":
    print(f"\nWARNING: no crash expected, but autocast is "
            f"{trainer.accelerator.mixed_precision!r}, not fp16. On a T4 that is the slow "
            "emulated path (~50s/step). Restart the kernel and run top-to-bottom.")
else:
    print("\nOK -> autocast is fp16 and all grads are fp32. Safe to train.")

In [ ]:
print("Training started...")
trainer.train()
print("Training loop finished.")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2}.


Training started...


Step,Training Loss,Validation Loss


In [ ]:
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print("Stage 2 training finished & LoRA adapters saved!")

# Free the training model + trainer before loading a fresh copy for inference below -
# otherwise two full model copies can exceed the T4's VRAM and cause silent load failures.
import gc
del trainer
del model
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after cleanup: {torch.cuda.memory_allocated() / 1024**3:.2f} GB allocated")

In [ ]:
# Load base model + Stage 2 LoRA adapter
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map=device_map,
    dtype=compute_dtype
)
base_model.config.use_cache = True  # re-enable KV cache for fast generation (was off for training)
model_with_adapter = PeftModelForCausalLM.from_pretrained(base_model, output_dir)
model_with_adapter.eval()

# Darija -> English
prompt_ar = AR_TO_EN_TEMPLATE.format(sentence="حاب نتعلم البرمجة.")
inputs = tokenizer(prompt_ar, return_tensors="pt").to("cuda")
with torch.inference_mode():
    outputs = model_with_adapter.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        do_sample=True
    )
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

# English -> Darija
prompt_en = EN_TO_AR_TEMPLATE.format(sentence="I want to learn programming.")
inputs = tokenizer(prompt_en, return_tensors="pt").to("cuda")
with torch.inference_mode():
    outputs = model_with_adapter.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        do_sample=True
    )
print(tokenizer.decode(outputs[0], skip_special_tokens=True))